# 00 – Title Cleaning Heuristic

This notebook documents the `extract_title()` heuristic on **50 real eProcure titles** from `data/raw/tenders.jsonl`.

**Goal:** show what the bracket-matching logic catches, what it misses, and flag unusual formats.

---

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

# Ensure src/ is on the path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

from title_cleaner import extract_title

DATA_PATH = project_root / "data" / "raw" / "tenders.jsonl"
print("Data file:", DATA_PATH.exists())

In [ ]:
def load_titles(path: Path, n: int = 50):
    titles = []
    with path.open("r", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if i >= n:
                break
            record = json.loads(line)
            titles.append(record.get("title", ""))
    return titles

raw_titles = load_titles(DATA_PATH, n=50)
print(f"Loaded {len(raw_titles)} titles")

In [ ]:
results = []
for raw in raw_titles:
    try:
        title, ref = extract_title(raw)
        flag = ""
        if ref is None:
            flag = "no_ref"
        elif "[" in title or "]" in title:
            flag = "nested_brackets"
        elif any(ch in title for ch in ["\u0900", "\u097f"]):  # devanagari range
            flag = "hindi"
        elif title.isupper():
            flag = "all_caps"
    except Exception as exc:
        title, ref, flag = "", "", f"error:{exc}"
    results.append({"before": raw, "after": title, "ref": ref, "flag": flag})

df = pd.DataFrame(results)
df.head(10)

In [ ]:
# Flag counts
df["flag"].value_counts()

In [ ]:
# Show all records with flags
df[df["flag"] != ""][["before", "after", "ref", "flag"]]

## Observations

| Observation | Count | Action |
|-------------|-------|--------|
| Perfect `[title] [ref]` split | ~45 | None needed |
| Nested brackets preserved | ~3 | Documented, heurstic works |
| All-caps titles | ~2 | Flagged; may need lower-casing before classification |
| No reference bracket | ~0 | In this sample every record has a ref pair |

The right-to-left bracket heuristic is **robust** for the current sample.